# Dự án Phân tích & Ước lượng Tiềm năng Doanh số (Sales Potential Estimation)

Notebook này thực hiện các bước thiết lập môi trường, lấy dữ liệu thô từ Kaggle, khảo sát phân phối dữ liệu, phân tích chất lượng dữ liệu (giá trị thiếu, ngoại lệ) và thực hiện tiền xử lý (làm sạch dữ liệu) cho bộ dữ liệu thương mại điện tử **Olist (Brazil)**.

## 1. Thiết lập Môi trường & Khai báo Thư viện
Nạp các thư viện phân tích số liệu (`pandas`, `numpy`), trực quan hóa (`matplotlib`, `seaborn`), và các cấu hình hệ thống cần thiết.


In [16]:
# ===========================
# TOÀN BỘ SETUP CHO NOTEBOOK
# ===========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

import sys
import os
import shutil
import warnings

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

from src.utils.system_utils import (
    project_root,
    print_section_header
)

ImportError: cannot import name 'get_web_endpoint' from 'kagglesdk.kaggle_env' (D:\FPT_Syllabus\DAP301m\DAP391m_AI2013_G7\.venv\lib\site-packages\kagglesdk\kaggle_env.py)

---
# GIỚI THIỆU DỰ ÁN
---

## 1.1 Bối cảnh thực tiễn

Mở rộng thị trường kinh doanh là một quyết định có rủi ro tài chính lớn do chi phí mặt bằng và vận hành cao. Cách tiếp cận truyền thống dựa trên trực giác hoặc số liệu nhân khẩu học cơ bản dễ dẫn đến sai lầm trong việc chọn vị trí (site selection). Các doanh nghiệp cần tận dụng dữ liệu hành vi mua sắm trực tuyến (E-commerce) để định vị chính xác "mật độ nhu cầu" thực tế của khách hàng, từ đó tối ưu hóa việc phân bổ hạ tầng vật lý offline.


## 1.2 Phát biểu bài toán

Dự án sử dụng bộ dữ liệu thương mại điện tử **Olist (Brazil)** làm bộ dữ liệu trung tâm cho giai đoạn thực nghiệm. Bài toán đặt ra là: Làm thế nào để tổng hợp dữ liệu giao dịch trực tuyến (vị trí khách hàng, doanh số, phí vận chuyển, thời gian giao hàng) kết hợp với dữ liệu nhân khẩu học bên ngoài (dân số, thu nhập) để xây dựng một mô hình ước lượng **Tiềm năng doanh số (Sales Potential)** tại từng khu vực, từ đó làm căn cứ khoa học cho quyết định mở rộng thị trường kinh doanh.


## 1.3 Mục tiêu dự án

-   **Phân tích tăng trưởng vùng:** Xác định các khu vực có tốc độ tăng trưởng doanh số trực tuyến nhanh nhất để khoanh vùng cơ hội.
-   **Xác định nhân tố ảnh hưởng:** Xây dựng mô hình hồi quy (Regression) để làm rõ các yếu tố (dân số, chi phí logistics, hành vi mua sắm) tác động mạnh nhất đến doanh số.
-   **Xếp hạng ưu tiên mở rộng:** Phát triển bộ chỉ số cơ hội (Opportunity Score) để xếp hạng các khu vực tiềm năng và đề xuất danh sách **Top 3 địa điểm** tối ưu nhất cho đội ngũ phát triển mặt bằng.

---
# MÔ TẢ DỮ LIỆU
---

## 2.1 Lấy dữ liệu

In [ ]:
print_section_header("LẤY DỮ LIỆU TỪ KAGGLE")


olist_raw_dir = project_root / "data/raw/olist/"

downloaded_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")


for file_name in os.listdir(downloaded_path):
    downloaded_file = os.path.join(downloaded_path, file_name)
    olist_dataset_file = os.path.join(olist_raw_dir, file_name)
    
    if os.path.isfile(downloaded_file):
        shutil.copy(downloaded_file, olist_dataset_file)
        print(f"Đã lưu: {olist_dataset_file}")

## 2.2 Tìm hiểu dữ liệu

In [ ]:
olist_customers = pd.read_csv(olist_raw_dir / 'olist_customers_dataset.csv')
olist_geolocation = pd.read_csv(olist_raw_dir / 'olist_geolocation_dataset.csv')
olist_orders = pd.read_csv(olist_raw_dir / 'olist_orders_dataset.csv')
olist_order_items = pd.read_csv(olist_raw_dir / 'olist_order_items_dataset.csv')
olist_order_payments = pd.read_csv(olist_raw_dir / 'olist_order_payments_dataset.csv')
olist_order_reviews = pd.read_csv(olist_raw_dir / 'olist_order_reviews_dataset.csv')
olist_products = pd.read_csv(olist_raw_dir / 'olist_products_dataset.csv')
olist_sellers = pd.read_csv(olist_raw_dir / 'olist_sellers_dataset.csv')


dict_olist_df = {
    'Customers'      : olist_customers,
    'Geolocation'    : olist_geolocation,
    'Orders'         : olist_orders,
    'Order Items'    : olist_order_items,
    'Order Payments' : olist_order_payments,
    'Order Reviews'  : olist_order_reviews,
    'Products'       : olist_products,
    'Sellers'        : olist_sellers,
}

for name, df in dict_olist_df.items():
    print_section_header("BẢNG " + name.upper())
    
    display(df.head())

    print("--- Kích thước ---")
    display(df.shape)
    
    print("--- Mô tả ---")
    display(df.describe())

---
# TIỀN XỬ LÝ DỮ LIỆU
---

## 3.1 Những việc cần làm

- Xử lý trùng lặp
- Xử lý Null/NaN
- Chuẩn hóa kiểu dữ liệu
- Kiểm tra categories
- Dữ liệu không hợp lệ
- Loại bỏ outliers

Xử lý những features nào??

## 3. Tiền xử lý dữ liệu

Trong phần này, chúng ta tiến hành:
1. Kiểm tra cấu trúc dữ liệu thô.
2. Xác định các vấn đề về chất lượng dữ liệu (Missing Values, Outliers).
3. Trực quan hóa phân phối của các biến số chính.
4. Thực hiện làm sạch dữ liệu và lưu lại phiên bản sạch phục vụ cho các bước phân tích tiếp theo.

### 3.1 Khởi tạo thư mục đích lưu trữ dữ liệu sau xử lý


In [ ]:
processed_olist_dir = project_root / "data/processed/olist/"

### 3.2 Khảo sát cấu trúc tổng quan của 8 bảng dữ liệu thô
In ra kích thước (`shape`), 3 dòng dữ liệu đầu (`head(3)`) và các chỉ số thống kê mô tả (`describe`) để có cái nhìn tổng quan về định dạng dữ liệu thô.


In [ ]:
# ============================================================
# In head() + describe() cho từng bảng
# ============================================================
for name, df in dict_olist_df.items():
    print(f'Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột')
    display(df.head(3))
    display(df.describe(include='all').T)

### 3.3 Phân tích tỷ lệ giá trị thiếu (Missing Values) theo từng bảng
Tổng hợp tất cả các cột có giá trị Null/NaN trong toàn bộ 8 bảng dữ liệu thô để xác định các cột cần được làm sạch hoặc loại bỏ.


In [ ]:
# ============================================================
# BẢNG TỔNG HỢP GIÁ TRỊ THIẾU
# ============================================================
# NOTE: Tổng hợp số lượng và tỉ lệ null cho toàn bộ 8 bảng
#       vào một DataFrame duy nhất để dễ nhìn.

missing_summary = []

for name, df in dict_olist_df.items():
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    for col in df.columns:
        if null_counts[col] > 0:
            missing_summary.append({
                'Bảng'          : name,
                'Cột'           : col,
                'Số giá trị null': int(null_counts[col]),
                'Tỉ lệ null (%)' : float(null_pct[col]),
            })

df_missing = pd.DataFrame(missing_summary).sort_values('Tỉ lệ null (%)', ascending=False)
print(f'Tổng số cột có giá trị null: {len(df_missing)}')
display(df_missing)

### 3.4 Trực quan hóa chất lượng dữ liệu
Vẽ biểu đồ so sánh tỉ lệ Null cao nhất giữa các bảng và top 10 cột có số lượng giá trị thiếu nhiều nhất để đưa ra chiến lược xử lý phù hợp.


In [ ]:
# ============================================================
# BIỂU ĐỒ: Heatmap giá trị null theo bảng & cột
# ============================================================
# NOTE: Ô màu sáng = có null. Giúp nhìn nhanh bảng nào và cột
#       nào cần ưu tiên xử lý.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Subplot 1: Tỉ lệ null theo từng bảng ---
null_by_table = (
    df_missing
    .groupby('Bảng')['Tỉ lệ null (%)']
    .max()
    .sort_values(ascending=False)
)

bars = axes[0].barh(null_by_table.index, null_by_table.values,
                    color=sns.color_palette('Reds_r', len(null_by_table)))
axes[0].set_xlabel('Tỉ lệ null tối đa (%)')
axes[0].set_title('Tỉ lệ NULL cao nhất theo Bảng')
for bar, val in zip(bars, null_by_table.values):
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

# --- Subplot 2: Số dòng null theo cột ---
top_null = df_missing.nlargest(10, 'Số giá trị null')
axes[1].barh(
    top_null['Bảng'] + ' · ' + top_null['Cột'],
    top_null['Số giá trị null'],
    color=sns.color_palette('Blues_r', len(top_null))
)
axes[1].set_xlabel('Số dòng null')
axes[1].set_title('Top 10 Cột có nhiều NULL nhất')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.suptitle('  Phân tích Chất lượng Dữ liệu — Missing Values', y=1.02, fontsize=14, fontweight='bold')
plt.show()

### 3.5 Khảo sát phân phối của Giá bán (`price`) & Phí vận chuyển (`freight_value`)
Sử dụng biểu đồ phân phối (Histogram) và biểu đồ hộp (Boxplot) để kiểm tra mức độ lệch (skewness) cũng như phát hiện các điểm dị biệt (outliers).


In [ ]:
# ============================================================
# BIỂU ĐỒ: Phân phối giá bán (price) & phí vận chuyển
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- 1) Histogram giá gốc ---
sns.histplot(olist_order_items['price'], bins=10, ax=axes[0][0])
axes[0][0].set_yscale('log')
axes[0][0].set_title('Phân phối Giá bán (toàn bộ)')
axes[0][0].set_xlabel('Giá (BRL)')
axes[0][0].set_ylabel('Số lượng')
axes[0][0].legend()

# --- 2) Histogram phí vận chuyển ---
axes[0][1].set_yscale('log')
sns.histplot(olist_order_items['freight_value'], bins=10, ax=axes[0][1])
axes[0][1].set_title(f'Phân phối Giá ship')
axes[0][1].set_xlabel('Giá (BRL)')
axes[0][1].set_ylabel('Số lượng')

# --- 3) Boxplot Giá bán ---
sns.boxplot(y=olist_order_items['price'], ax=axes[1][0])
axes[1][0].set_title('Boxplot Giá bán')
axes[1][0].set_ylabel('Giá (BRL)')
axes[1][0].legend()

# --- 4) Boxplot Phí vận chuyển ---
sns.boxplot(y=olist_order_items['freight_value'], ax=axes[1][1])
axes[1][1].set_title('Boxplot Phí vận chuyển (freight_value)')
axes[1][1].set_ylabel('Phí vận chuyển (BRL)')

plt.tight_layout()
plt.suptitle('Phân phối Giá bán & Phí vận chuyển', y=1.02, fontsize=14, fontweight='bold')
plt.show()

### 3.6 Thực hiện Log Transformation cho các biến số lệch nhiều
Do phân phối của `price` và `freight_value` có độ lệch rất cao (hầu hết tập trung ở vùng giá thấp nhưng có các ngoại lệ cực kỳ cao), ta sử dụng hàm biến đổi $\log(x + 1)$ để phân phối tiệm cận phân phối chuẩn, hỗ trợ tốt hơn cho các mô hình hồi quy sau này.


In [ ]:
# ============================================================
# BIỂU ĐỒ: Phân phối giá bán (price) & phí vận chuyển
# ============================================================
import numpy as np

# Log transform (log1p để tránh log(0))
price_log   = np.log1p(olist_order_items['price'])
freight_log = np.log1p(olist_order_items['freight_value'])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- 1) Histogram giá (log) ---
sns.histplot(price_log, bins=80, kde=True, ax=axes[0][0])
axes[0][0].set_title('Phân phối Giá bán (log scale)')
axes[0][0].set_xlabel('log(Giá + 1)')
axes[0][0].set_ylabel('Số lượng')

# --- 2) Histogram phí vận chuyển (log) ---
sns.histplot(freight_log, bins=80, kde=True, ax=axes[0][1])
axes[0][1].set_title('Phân phối Phí vận chuyển (log scale)')
axes[0][1].set_xlabel('log(Phí ship + 1)')
axes[0][1].set_ylabel('Số lượng')

# --- 3) Boxplot giá (log) ---
sns.boxplot(y=price_log, ax=axes[1][0])
axes[1][0].set_title('Boxplot Giá bán (log scale)')
axes[1][0].set_ylabel('log(Giá + 1)')

# --- 4) Boxplot phí vận chuyển (log) ---
sns.boxplot(y=freight_log, ax=axes[1][1])
axes[1][1].set_title('Boxplot Phí vận chuyển (log scale)')
axes[1][1].set_ylabel('log(Phí ship + 1)')

plt.tight_layout()
plt.suptitle('  Phân phối Giá bán & Phí vận chuyển (sau Log Transform)',
             y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Thống kê so sánh
print('  Giá bán (gốc):')
print(olist_order_items['price'].describe().round(2))
print('\n  Giá bán (sau log):')
print(price_log.describe().round(2))

### 3.7 Kiểm tra lại bảng tổng hợp missing values trước khi thực hiện Data Cleaning


In [ ]:
# ============================================================
# BẢNG TỔNG HỢP GIÁ TRỊ THIẾU
# ============================================================
# NOTE: Tổng hợp số lượng và tỉ lệ null cho toàn bộ 8 bảng
#       vào một DataFrame duy nhất để dễ nhìn.

missing_summary = []

for name, df in dict_olist_df.items():
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    for col in df.columns:
        if null_counts[col] > 0:
            missing_summary.append({
                'Bảng'          : name,
                'Cột'           : col,
                'Số giá trị null': int(null_counts[col]),
                'Tỉ lệ null (%)' : float(null_pct[col]),
            })

df_missing = pd.DataFrame(missing_summary).sort_values('Tỉ lệ null (%)', ascending=False)
print(f'Tổng số cột có giá trị null: {len(df_missing)}')
display(df_missing)

### 3.8 Làm sạch dữ liệu bảng `olist_orders`
*   Chỉ giữ lại các đơn hàng đã giao thành công (`order_status == 'delivered'`).
*   Bỏ cột `order_approved_at` do không cần thiết cho mô hình logistic/sales potential.
*   Loại bỏ các bản ghi bị thiếu thông tin ngày giao cho đơn vị vận chuyển (`order_delivered_carrier_date`) và ngày giao cho khách hàng (`order_delivered_customer_date`).


In [ ]:
olist_orders_clean = (
    olist_orders[olist_orders['order_status'] == 'delivered']
    .drop(columns=['order_approved_at'])
    .dropna(subset=['order_delivered_carrier_date', 'order_delivered_customer_date'])
    .copy()
)
olist_orders_clean.to_csv(processed_olist_dir + 'orders.csv', index=False)
print(f'   orders: {len(olist_orders_clean):,} dòng (chỉ delivered)')

### 3.9 Làm sạch dữ liệu bảng `olist_products`
Loại bỏ các dòng bị thiếu danh mục sản phẩm (`product_category_name`) hoặc thiếu thông tin kích thước/trọng lượng sản phẩm.


In [ ]:
olist_products_clean = (
    olist_products
    .dropna(subset=['product_category_name',
                    'product_weight_g',
                    'product_length_cm',
                    'product_height_cm',
                    'product_width_cm'])
    .copy()
)

olist_products_clean.to_csv(processed_olist_dir + 'products.csv', index=False)
print(f'  products: {len(olist_products_clean):,} dòng (đã xóa null category)')


### 3.10 Làm sạch dữ liệu bảng `olist_order_reviews`
Bỏ các cột chứa nội dung bình luận chi tiết (`review_comment_title`, `review_comment_message`) để giảm kích thước lưu trữ, chỉ giữ lại điểm số đánh giá (`review_score`) và ID đơn hàng phục vụ cho phân tích định lượng.


In [ ]:
olist_reviews_clean = (
    olist_order_reviews
    .drop(columns=['review_comment_title', 'review_comment_message'])
    .copy()
)

olist_reviews_clean.to_csv(processed_olist_dir + 'order_reviews.csv', index=False)
print(f'  order_reviews: dropped title & message columns')

### 3.11 Đánh giá kết quả làm sạch dữ liệu
So sánh số lượng dòng trước và sau khi làm sạch đối với các bảng quan trọng để kiểm soát lượng dữ liệu bị loại bỏ và đảm bảo không còn giá trị Null.


In [ ]:

print(f'  orders   : {len(olist_orders):,} → {len(olist_orders_clean):,} dòng')
print(f'  products : {len(olist_products):,} → {len(olist_products_clean):,} dòng')
print(f'  reviews  : {olist_order_reviews.shape[1]} cols → {olist_reviews_clean.shape[1]} cols')

print('\n  Null còn lại:')
for name, df in [('orders', olist_orders_clean),
                 ('products', olist_products_clean),
                 ('reviews', olist_reviews_clean)]:
    n = df.isnull().sum().sum()
    print(f'  {name:12s} → {"  sạch" if n == 0 else f"⚠️ {n} nulls"}')

### 3.12 Xuất các bảng dữ liệu đã làm sạch ra file CSV
Lưu các bảng `customers`, `order_payments`, `sellers` sang định dạng sạch trong thư mục dữ liệu đã tiền xử lý.


In [ ]:
olist_customers.to_csv(processed_olist_dir      + 'customers.csv',       index=False)
olist_order_payments.to_csv(processed_olist_dir + 'order_payments.csv',   index=False)
olist_sellers.to_csv(processed_olist_dir        + 'sellers.csv',          index=False)